# 02 — Preprocessing & closest-users features (Step 2)
Build the uniform ACC Arena table with **all 12,000 users** by default, engineer Team-8 closest-user features, split by user, and fit preprocessing statistics on training users only. The primary dataset retains every traffic state and the full throughput distribution; optional scope restrictions are disabled and explicitly labeled as sensitivity analyses.

In [ ]:
# === Project config — Team 8: Throughput Prediction in a Dense 5G deployment (with Transfer Learning) ===
RANDOM_SEED      = 42
RESAMPLE_SECONDS = 120           # time granularity: THE dataset-size knob (professor's guidance: coarsen the
                                 # time grid rather than subsample users). Raw cadence is ~3s with frequent 4s
                                 # steps, duplicate stamps and gaps up to 7s (ACC) / ~1s (Salt&Tar); each output row aggregates every raw sample in a fixed-width window.
N_USERS          = None          # ACC Arena users. None = ALL (~12k), so the X-closest neighbourhoods are the
                                 # real ones; an int (e.g. 1500, seeded random sample) biases the neighbourhoods
                                 # (X closest searched within the sample) and is only for quick debug runs.
N_USERS_SALT     = 3000          # Salt&Tar users: ALL of them (only ~1/3 are ever active; activity is id-biased)
X_VALUES         = [3, 5, 10]    # number of closest users to experiment with. X=1 excluded by design:
                                 # heavy co-location makes a single arbitrary neighbour uninformative.
                                 # X=0 (no neighbour features) IS produced and trained: it is the baseline
                                 # that quantifies what the closest-users features actually add.
ENCODINGS        = ["pos", "agg"]# neighbour-feature encodings: "pos" = ordered per-neighbour columns
                                 # (nb0_*, nb1_*, ...), "agg" = order-invariant aggregates over the X
                                 # neighbours (sum/mean/count) — rationale in notebook 02.
BEST_X           = 3             # X used for the transfer-learning experiment (pick the best from notebook 04)
BEST_ENC         = "pos"         # encoding used for the transfer-learning experiment (pick from notebook 04)
OUTLIER_PCT      = None          # optional train-only sensitivity analysis; primary results keep the full distribution.
                                 # EDA (notebook 01): the ~1% extreme samples carry ~2/3 of the total variance.
ACTIVE_ONLY      = False          # primary task covers every user; True is an optional active-only sensitivity run
MIN_TRAFFIC_TYPE = 2             # keep traffic_type >= this (2=const_rate, 3=video, 4=gaming, 5=http)

# --- data location ---
DRIVE_FOLDER_ID  = "1s1YCWyhN_Fv5sQraTVs4Rga-ATiKPRgo"   # shared Drive folder containing the zip
ZIP_NAME         = "L5GHDD_Dataset.zip"
DATA_ROOT        = "data/raw"     # dataset folders live here after unzip (local default)
PROCESSED_DIR    = "data/processed"
RESULTS_DIR      = "results"

import os, numpy as np, random
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: mount Drive and make outputs PERSIST across notebooks (no-op locally) ===
def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR   = "/content/drive/MyDrive/5g_throughput_team8"   # persistent project folder on your Drive
    PROCESSED_DIR = f"{PROJECT_DIR}/processed"                     # 02 writes here, 03/04/05 read from here
    RESULTS_DIR   = f"{PROJECT_DIR}/results"                       # models, metrics.csv, figures
    print("Outputs persist in:", PROJECT_DIR)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: locate and unzip the dataset (no-op locally) ===
if in_colab():
    import glob, zipfile, subprocess
    DATA_ROOT = "/content/L5GHDD_Dataset"
    if not os.path.isdir(DATA_ROOT):
        cands = glob.glob(f"/content/drive/MyDrive/**/{ZIP_NAME}", recursive=True)
        if not cands:                                   # fallback: download the shared folder
            subprocess.run(["pip", "-q", "install", "gdown"], check=False)
            import gdown
            gdown.download_folder(id=DRIVE_FOLDER_ID, output="/content/_dl", quiet=False, use_cookies=False)
            cands = glob.glob(f"/content/_dl/**/{ZIP_NAME}", recursive=True)
        assert cands, f"{ZIP_NAME} not found. Put it in your Drive or share the folder."
        print("Unzipping", cands[0], "...")
        with zipfile.ZipFile(cands[0]) as z:
            z.extractall(DATA_ROOT)
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
# === Data loading helpers (raw wide format -> uniform tidy windows) ===
import glob, re, math
import pandas as pd
from scipy.spatial import cKDTree

def find_venue_dir(data_root, venue_key):
    """venue_key in {'acc','salt'}; robust to the zip's internal layout."""
    pat = {"acc": "*ACC*Arena*", "salt": "*Salt*Tar*"}[venue_key]
    hits = [os.path.join(dp, d) for dp, dn, _ in os.walk(data_root)
            for d in dn if glob.fnmatch.fnmatch(d, pat)]
    hits = sorted(set(hits), key=len)
    assert hits, f"venue '{venue_key}' not found under {data_root}"
    return hits[0]

def file_id_range(path):
    m = re.search(r'_(\d+)_(\d+)\.csv$', path)
    return int(m.group(1)), int(m.group(2))

def metric_files(venue_dir, subdir_glob, file_glob, user_ids=None):
    subs = glob.glob(os.path.join(venue_dir, subdir_glob))
    assert subs, f"no subdir matching {subdir_glob} in {venue_dir}"
    files = sorted(glob.glob(os.path.join(subs[0], file_glob)), key=lambda p: file_id_range(p)[0])
    if user_ids is not None:
        def overlaps(path):
            first, last = file_id_range(path)
            return any(first <= user <= last for user in user_ids)
        files = [path for path in files if overlaps(path)]
    assert files, f"no files matching {file_glob} in {subdir_glob}"
    return files

def all_user_ids(venue_dir):
    ids = []
    for path in metric_files(venue_dir, "*Throughput*", "*.csv"):
        first, last = file_id_range(path)
        ids.extend(range(first, last + 1))
    return np.asarray(sorted(ids))

def _time_origin(venue_dir):
    """Common origin shared by both raw timelines in a venue."""
    starts = []
    for path in glob.glob(os.path.join(venue_dir, "**", "*.csv"), recursive=True):
        starts.append(float(pd.read_csv(path, usecols=[0], nrows=1).iloc[0, 0]))
    return min(starts)

def _window_index(times, origin, seconds):
    return origin + np.floor((np.asarray(times, dtype=float) - origin) / seconds) * seconds

def load_metric(files, value_name, origin, seconds, user_ids, how="mean"):
    """Aggregate every raw observation into a deterministic, uniform time window."""
    out = []
    for path in files:
        header = list(pd.read_csv(path, nrows=0).columns)
        column_to_user = {c: int(re.search(r'(\d+)', c).group(1)) for c in header[1:]}
        usecols = [header[0]] + [c for c in header[1:] if column_to_user[c] in user_ids]
        wide = pd.read_csv(path, usecols=usecols).rename(columns={header[0]: "raw_time"})
        wide["time"] = _window_index(wide.pop("raw_time"), origin, seconds)
        values = wide.groupby("time", sort=True)
        wide = (values.mean() if how == "mean" else values.last()).astype("float32")
        wide = wide.rename(columns=column_to_user)
        out.append(wide.reset_index().melt(id_vars="time", var_name="user_id", value_name=value_name))
    return pd.concat(out, ignore_index=True)

def load_positions(files, origin, seconds, user_ids):
    """Aggregate positions per window and convert latitude/longitude to local metres."""
    frames = []
    for path in files:
        first = pd.read_csv(path, nrows=1).values.astype(float)
        all_ids = first[0, 1::5].astype(int)
        blocks = [k for k, user in enumerate(all_ids) if user in user_ids]
        if not blocks:
            continue
        usecols = [0] + [1 + 5 * k + j for k in blocks for j in range(5)]
        raw = pd.read_csv(path, usecols=usecols).values.astype(float)
        ids = raw[0, 1::5].astype(int)
        bins = _window_index(raw[:, 0], origin, seconds)
        pieces = []
        for name, values, how in [
            ("lat", raw[:, 2::5], "mean"), ("lon", raw[:, 3::5], "mean"),
            ("z", raw[:, 4::5], "mean"), ("traffic_type", raw[:, 5::5], "last")]:
            wide = pd.DataFrame(values, index=bins, columns=ids)
            wide = wide.groupby(level=0, sort=True)
            wide = wide.mean() if how == "mean" else wide.last()
            wide.index.name = "time"
            pieces.append(wide.reset_index().melt(id_vars="time", var_name="user_id", value_name=name))
        frame = pieces[0]
        for piece in pieces[1:]:
            frame = frame.merge(piece, on=["time", "user_id"], validate="one_to_one")
        frames.append(frame)
    pos = pd.concat(frames, ignore_index=True)
    lat0, lon0 = pos.lat.mean(), pos.lon.mean()
    radius_m = 6_371_000.0
    pos["x"] = radius_m * np.radians(pos.lon - lon0) * math.cos(math.radians(lat0))
    pos["y"] = radius_m * np.radians(pos.lat - lat0)
    return pos[["time", "user_id", "x", "y", "z", "traffic_type"]]

def assemble(venue_key, n_users, resample_seconds, random_users=False):
    venue = find_venue_dir(DATA_ROOT, venue_key)
    population = all_user_ids(venue)
    if n_users is None:
        user_ids = set(map(int, population))
        print(f"{venue_key}: using ALL {len(user_ids)} users")
    elif random_users:
        rng = np.random.default_rng(RANDOM_SEED)
        user_ids = set(map(int, rng.choice(population, size=min(n_users, len(population)), replace=False)))
        print(f"{venue_key}: sampled {len(user_ids)} random users out of {len(population)}")
    else:
        user_ids = set(map(int, population[:n_users]))
    origin = _time_origin(venue)
    mf = lambda sub, pattern: metric_files(venue, sub, pattern, user_ids)
    parts = [
        load_metric(mf("*Throughput*", "*.csv"), "throughput", origin, resample_seconds, user_ids),
        load_metric(mf("*BLER*", "*.csv"), "bler", origin, resample_seconds, user_ids),
        load_metric(mf("*PRB*", "*.csv"), "prb", origin, resample_seconds, user_ids),
        load_metric(mf("*RU_Association*", "*.csv"), "ru_id", origin, resample_seconds, user_ids, how="last"),
        load_metric(mf("*SINR*", "SINRDL_*.csv"), "sinr_dl", origin, resample_seconds, user_ids),
        load_metric(mf("*SINR*", "SINRUL_*.csv"), "sinr_ul", origin, resample_seconds, user_ids),
        load_positions(mf("*Positions*", "*.csv"), origin, resample_seconds, user_ids),
    ]
    frame = parts[0]
    for part in parts[1:]:
        frame = frame.merge(part, on=["time", "user_id"], how="inner", validate="one_to_one")
    frame = frame.dropna().reset_index(drop=True)
    frame["user_id"] = frame.user_id.astype(int)
    frame["traffic_type"] = frame.traffic_type.round().astype(int)
    frame["ru_id"] = frame.ru_id.round().astype(int)
    assert not frame.duplicated(["time", "user_id"]).any()
    return frame


In [ ]:
# === Closest-users feature engineering (Team-8 specific) ===
# The target variable is deliberately excluded: neighbour throughput would leak labels across user splits.
NEIGHBOR_FEATS = ["sinr_dl", "sinr_ul", "prb", "bler", "traffic_type"]

def add_closest_user_features(df, x_max):
    cols = []
    for k in range(x_max):
        cols += [f"nb{k}_dist"] + [f"nb{k}_{c}" for c in NEIGHBOR_FEATS]
    out = np.full((len(df), len(cols)), np.nan, dtype="float32")
    feat = df[NEIGHBOR_FEATS].values
    pos = df[["x", "y", "z"]].values
    for _, idx in df.groupby("time", sort=False).groups.items():
        idx = np.asarray(idx)
        n = len(idx)
        if n <= 1:
            continue
        k = min(x_max + 1, n)
        dist, neighbours = cKDTree(pos[idx]).query(pos[idx], k=k)
        if k == 1:
            dist, neighbours = dist[:, None], neighbours[:, None]
        rows = np.arange(n)[:, None]
        order = np.argsort(neighbours == rows, axis=1, kind="stable")
        take = min(x_max, k - 1)
        r = np.repeat(np.arange(n), take)
        c = order[:, :take].ravel()
        block = np.empty((n, take, 1 + len(NEIGHBOR_FEATS)), dtype="float32")
        block[:, :, 0] = dist[r, c].reshape(n, take)
        block[:, :, 1:] = feat[idx[neighbours[r, c]]].reshape(n, take, -1)
        out[idx, :take * (1 + len(NEIGHBOR_FEATS))] = block.reshape(n, -1)
    return pd.concat([df, pd.DataFrame(out, columns=cols, index=df.index)], axis=1)

def neighbor_cols(x):
    return [name for k in range(x)
            for name in [f"nb{k}_dist"] + [f"nb{k}_{feature}" for feature in NEIGHBOR_FEATS]]


In [ ]:
# === Order-invariant neighbour aggregates (encoding "agg") ===
AGG_FEATS = ["nb_prb_sum", "nb_sinr_dl_mean", "nb_sinr_ul_mean", "nb_bler_mean",
             "nb_active_count", "nb_distance_mean"]

def aggregate_neighbor_features(frame, x):
    def block(feature):
        return frame[[f"nb{k}_{feature}" for k in range(x)]]
    prb = block("prb")
    traffic = block("traffic_type")
    return pd.DataFrame({
        "nb_prb_sum": prb.sum(axis=1, min_count=1),
        "nb_sinr_dl_mean": block("sinr_dl").mean(axis=1),
        "nb_sinr_ul_mean": block("sinr_ul").mean(axis=1),
        "nb_bler_mean": block("bler").mean(axis=1),
        "nb_active_count": (traffic >= MIN_TRAFFIC_TYPE).sum(axis=1).astype(float),
        "nb_distance_mean": block("dist").mean(axis=1),
    }, index=frame.index)


## Build the full-population table and neighbour features
Neighbours are computed once at `max(X_VALUES)` over the full population present in each time window. This preserves the actual spatial context. The primary run keeps all target users; an optional active-only sensitivity filter is applied only after neighbour construction.

In [ ]:
import time as _time
t0 = _time.time()
df = assemble("acc", n_users=N_USERS, resample_seconds=RESAMPLE_SECONDS, random_users=True)
print("tidy shape:", df.shape, "| users:", df.user_id.nunique(), "| timestamps:", df.time.nunique())
df = add_closest_user_features(df, x_max=max(X_VALUES))
print("with neighbour features:", df.shape, "| built in %.1fs" % (_time.time()-t0))

if ACTIVE_ONLY:
    before = len(df)
    df = df[df.traffic_type >= MIN_TRAFFIC_TYPE].reset_index(drop=True)
    print(f"ACTIVE_ONLY: kept {len(df)}/{before} rows (traffic_type >= {MIN_TRAFFIC_TYPE})")
print("traffic types kept:", sorted(df.traffic_type.unique()))

## Split by user (train / val / test = 70 / 15 / 15)
Splitting on users avoids leaking a user's samples across splits.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
users = df.user_id.unique(); rng.shuffle(users)
n = len(users); tr = users[:int(.7*n)]; va = users[int(.7*n):int(.85*n)]; te = users[int(.85*n):]
split = {u:"train" for u in tr}; split.update({u:"val" for u in va}); split.update({u:"test" for u in te})
df["split"] = df.user_id.map(split)
print(df.split.value_counts())

## Define evaluation tail and optional train-only sensitivity
The training-set p99 is stored only to report slice metrics later; it is **not a removal rule**. `OUTLIER_PCT=None` is the required primary run. If a sensitivity threshold is explicitly enabled, it removes training rows only—validation and test remain complete and comparable.

In [ ]:
TRAIN_TAIL_P99 = float(np.percentile(df.loc[df.split == "train", "throughput"], 99))
print(f"Train p99 used only for evaluation slices: {TRAIN_TAIL_P99:.3f} Mbps")
if OUTLIER_PCT is not None:
    threshold = float(np.percentile(df.loc[df.split == "train", "throughput"], OUTLIER_PCT))
    remove = (df.split == "train") & (df.throughput > threshold)
    print(f"SENSITIVITY ONLY — train p{OUTLIER_PCT} threshold={threshold:.3f} Mbps; "
          f"removed {int(remove.sum())} training rows; validation/test unchanged")
    df = df.loc[~remove].reset_index(drop=True)

## Feature matrix builder
The schema is fixed across venues: own BLER/PRB/SINR/position, one-hot traffic state, and the selected neighbour encoding. Neighbour throughput is excluded. Missing-value medians and scaling parameters are fitted on training users only and reused unchanged for validation, test, and Salt&Tar.

In [ ]:
from sklearn.preprocessing import StandardScaler
import json

BASE_NUM = ["bler", "prb", "sinr_dl", "sinr_ul", "x", "y", "z"]
TRAFFIC_CLASSES = [0, 1, 2, 3, 4, 5]

def onehot_traffic(frame):
    return pd.DataFrame({f"traffic_{c}": (frame.traffic_type == c).astype(float)
                         for c in TRAFFIC_CLASSES}, index=frame.index)

def build_matrix(frame, x, scaler=None, medians=None, fit=False, enc="pos"):
    """Fit every preprocessing statistic on train only, then reuse it unchanged."""
    if enc == "agg" and x > 0:
        frame = pd.concat([frame, aggregate_neighbor_features(frame, x)], axis=1)
        num_cols = BASE_NUM + AGG_FEATS
    else:
        num_cols = BASE_NUM + neighbor_cols(x)
    numeric = frame[num_cols].astype("float32")
    if fit:
        medians = numeric.median()
        scaler = StandardScaler().fit(numeric.fillna(medians))
    assert scaler is not None and medians is not None
    numeric = pd.DataFrame(scaler.transform(numeric.fillna(medians)),
                           columns=num_cols, index=frame.index)
    matrix = pd.concat([numeric, onehot_traffic(frame)], axis=1)
    return matrix.values.astype("float32"), list(matrix.columns), scaler, medians


## Save every X × encoding scenario and its evaluation metadata
Each artifact contains train/validation/test arrays, user groups, test traffic states, and the training p99 used only for slice reporting. The accompanying preprocessor contains train-fitted medians and scaler.

In [ ]:
import joblib
train_mask, val_mask, test_mask = df.split == "train", df.split == "val", df.split == "test"

def scenarios():
    yield 0, "none", "acc_X0"
    for x in X_VALUES:
        for enc in ENCODINGS:
            yield x, enc, f"acc_X{x}" + ("_agg" if enc == "agg" else "")

saved = []
for x, enc, stem in scenarios():
    Xtr, cols, scaler, medians = build_matrix(df[train_mask], x, fit=True, enc=enc)
    Xva, _, _, _ = build_matrix(df[val_mask], x, scaler=scaler, medians=medians, enc=enc)
    Xte, _, _, _ = build_matrix(df[test_mask], x, scaler=scaler, medians=medians, enc=enc)
    np.savez_compressed(
        f"{PROCESSED_DIR}/{stem}.npz",
        X_train=Xtr, y_train=df.loc[train_mask, "throughput"].to_numpy("float32"),
        groups_train=df.loc[train_mask, "user_id"].to_numpy("int32"),
        X_val=Xva, y_val=df.loc[val_mask, "throughput"].to_numpy("float32"),
        groups_val=df.loc[val_mask, "user_id"].to_numpy("int32"),
        X_test=Xte, y_test=df.loc[test_mask, "throughput"].to_numpy("float32"),
        groups_test=df.loc[test_mask, "user_id"].to_numpy("int32"),
        traffic_test=df.loc[test_mask, "traffic_type"].to_numpy("int8"),
        train_tail_p99=np.asarray(TRAIN_TAIL_P99, dtype="float32"),
        active_only=np.asarray(ACTIVE_ONLY),
        outlier_pct=np.asarray(np.nan if OUTLIER_PCT is None else OUTLIER_PCT, dtype="float32"),
    )
    joblib.dump({"scaler": scaler, "medians": medians}, f"{PROCESSED_DIR}/{stem}_preprocessor.pkl")
    with open(f"{PROCESSED_DIR}/{stem}_cols.json", "w") as handle:
        json.dump(cols, handle)
    saved.append((x, enc, Xtr.shape[1], len(Xtr)))
    print(f"X={x:>2} enc={enc:<4} features={Xtr.shape[1]:>3} train={len(Xtr)} -> {stem}.npz")
print("saved:", saved)


Artifacts are now ready for notebook 03. Do not mix them with arrays generated by the older active-only/p99-filtered schema.